<a href="https://colab.research.google.com/github/CreatorPoints/PhotonCoreV2/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Super Wav2Lip

This Colab project is based on [Wav2Lip-GFPGAN](https://github.com/ajay-sainy/Wav2Lip-GFPGAN), updated for Python 3.12 / NumPy 2.x compatibility.

In [ ]:
import sys, builtins, site, glob, os
import numpy as np

# 1. Inject missing NumPy 1.x aliases into Python builtins & numpy
np.float = float
np.complex = complex
np.int = int
np.bool = bool

builtins.float32 = np.float32
builtins.complex128 = np.complex128

# 2. Safely patch installed librosa source files on disk
try:
    site_packages = site.getsitepackages()[0]
    librosa_files = glob.glob(os.path.join(site_packages, "librosa", "**", "*.py"), recursive=True)
    for filepath in librosa_files:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        
        # Fix specific deprecated aliases without breaking float32 keywords
        content = content.replace("np.complex", "complex")
        content = content.replace("dtype=float32", "dtype=np.float32")
        content = content.replace("np.float", "float")
        content = content.replace("np.int", "int")
        
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
except Exception as e:
    print("Patching warning:", e)

print("✅ System-wide librosa & NumPy environment patched successfully!")

## 1. Installation

Run this block to clone the repository and download pre-trained models.

In [ ]:
!git clone https://github.com/indianajson/wav2lip-HD.git
basePath = "/content/wav2lip-HD"
%cd {basePath}

wav2lipFolderName = 'Wav2Lip-master'
gfpganFolderName = 'GFPGAN-master'
wav2lipPath = basePath + '/' + wav2lipFolderName
gfpganPath = basePath + '/' + gfpganFolderName

!mkdir -p {wav2lipPath}/face_detection/detection/sfd/
!mkdir -p {wav2lipPath}/checkpoints/

!wget 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O {wav2lipPath}'/face_detection/detection/sfd/s3fd.pth'
!wget 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth' -O {wav2lipPath}'/checkpoints/wav2lip_gan.pth'
!wget 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip.pth' -O {wav2lipPath}'/checkpoints/wav2lip.pth'

!mkdir -p inputs
!mkdir -p outputs

!cd $gfpganFolderName && python setup.py develop
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P {gfpganFolderName}'/experiments/pretrained_models'

%cd {basePath}

from IPython.display import clear_output
clear_output()

print("Installation complete.")

## 2. Synchronize Video and Speech

In [ ]:
import os, sys, builtins
import numpy as np

# Guarantee global dtype availability
np.complex = complex
np.float = float
np.int = int
builtins.float32 = np.float32
builtins.complex128 = np.complex128

basePath = "/content/wav2lip-HD"
wav2lipFolderName = "/content/wav2lip-HD/Wav2Lip-master"

outputPath = basePath + '/outputs'
inputAudio = 'bruh.mp3' #@param{type:"string"}
inputAudioPath = basePath + '/inputs/' + inputAudio
inputVideo = 'guy.mp4' #@param{type:"string"}
inputVideoPath = basePath + '/inputs/' + inputVideo
lipSyncedOutputPath = basePath + '/outputs/result.mp4'
model = "wav2lip" #@param ["wav2lip", "wav2lip_gan"] {type:"string"}

if not os.path.exists(outputPath):
    os.makedirs(outputPath)

# Safe patch of Wav2Lip local audio.py
audio_py = os.path.join(wav2lipFolderName, "audio.py")
if os.path.exists(audio_py):
    with open(audio_py, "r", encoding="utf-8") as f:
        c = f.read()
    c = c.replace("np.complex", "complex").replace("np.float", "float")
    with open(audio_py, "w", encoding="utf-8") as f:
        f.write(c)

os.chdir(wav2lipFolderName)
sys.argv = [
    'inference.py',
    '--checkpoint_path', f'checkpoints/{model}.pth',
    '--face', inputVideoPath,
    '--audio', inputAudioPath,
    '--outfile', lipSyncedOutputPath
]

print("🚀 Running Wav2Lip synthesis...")
exec(open('inference.py').read())
print("✅ DONE! Check /content/wav2lip-HD/outputs/result.mp4")

## 3. Boost the Resolution of the Synthesized Video

In [ ]:
import cv2, os
from tqdm import tqdm
from os import path

inputVideoPath = outputPath + '/result.mp4'
unProcessedFramesFolderPath = outputPath + '/frames'

if not os.path.exists(unProcessedFramesFolderPath):
    os.makedirs(unProcessedFramesFolderPath)

vidcap = cv2.VideoCapture(inputVideoPath)
numberOfFrames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = vidcap.get(cv2.CAP_PROP_FPS)
print("FPS:", fps, "Frames:", numberOfFrames)

for frameNumber in tqdm(range(numberOfFrames)):
    _, image = vidcap.read()
    if image is not None:
        cv2.imwrite(path.join(unProcessedFramesFolderPath, str(frameNumber).zfill(4) + '.jpg'), image)

!cd $gfpganFolderName && python inference_gfpgan.py -i $unProcessedFramesFolderPath -o $outputPath -v 1.3 -s 2 --only_center_face --bg_upsampler None

restoredFramesPath = outputPath + '/restored_imgs/'
processedVideoOutputPath = outputPath

if not os.path.exists(restoredFramesPath):
    os.makedirs(restoredFramesPath)

dir_list = os.listdir(restoredFramesPath)
dir_list.sort()

inputVideoPath = outputPath + '/result.mp4'
vidcap = cv2.VideoCapture(inputVideoPath)
fps = vidcap.get(cv2.CAP_PROP_FPS)
print("The video is " + str(fps) + " FPS.")

batch = 0
batchSize = 1300
for i in tqdm(range(0, len(dir_list), batchSize)):
    img_array = []
    start, end = i, i + batchSize
    for filename in tqdm(dir_list[start:end]):
        filename = restoredFramesPath + filename
        img = cv2.imread(filename)
        if img is None:
            continue
        height, width, layers = img.shape
        size = (width, height)
        img_array.append(img)
    out = cv2.VideoWriter(processedVideoOutputPath + '/output_' + str(batch).zfill(4) + '.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, size)
    batch += 1

    for k in range(len(img_array)):
        out.write(img_array[k])
    out.release()

from IPython.display import clear_output
clear_output()

print("Video upscaling complete!")

## 4. Clear Cached Files

In [ ]:
%cd /content/wav2lip-HD/

removeInputs = True #@param {type:"boolean"}
removeOutputs = True #@param {type:"boolean"}

if removeInputs:
    !rm -rf inputs/*
if removeOutputs:
    !rm -rf outputs/frames/*
    !rm -rf outputs/restored_imgs/*
    !rm -rf outputs/*

from IPython.display import clear_output
clear_output()

print("Cleared cached files.")